Support Vector Machine Analysis
==============================================================
Research Questions addressed:

  Q4.   Which risk factor has a more significant impact on hypertension risk: cholesterol or exercise?

  Q5.   Which is a better predictor of hypertension risk: physical activity or time spent sitting?
  
  Q7.   Are income and education good predictors of hypertension risk?



Install libraries (as needed)

In [59]:
# %pip install scikit-learn # <- uncomment to install scikit-learn if needed

Import libraries

In [60]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 175)
pd.set_option("display.float_format", lambda x: f'{x:,.4f}')


Load dataset

In [61]:
ht_risk_df = pd.read_csv('../data/nhanes_hypertension_risk.csv')
ht_risk_df

,participant_id,age,race_ethnicity,education,poverty_income_ratio,marital_status,diagnosed_high_bp,diagnosed_twice,on_chol_medication,bmi,waist_cm,weight_kg,height_cm,smoked_100_cigarettes,smoke_frequency,avg_drinks_per_day,time_sitting,total_cholesterol_mgdl,HDL_cholesterol_mgdl,fasting_glucose_mgdl,LDL_cholesterol_mgdl_friedewald,LDL_cholesterol_mgdl_martin,LDL_cholesterol_mgdl_nih,systolic_avg,systolic_clinical,diastolic_avg,diastolic_clinical,pulse_avg,pulse_clinical,hypertension_risk,male,ever_smoker,current_smoker,drinks_alcohol,drink_frequency_past_year,high_cholesterol,moderate_minutes_per_week,vigorous_minutes_per_week,physically_active
0,130378,43.0000,Non-Hispanic Asian,College graduate or above,5.0000,Married/Living with Partner,1,1.0000,No,27.0000,98.3000,86.9000,179.5000,1.0000,3.0000,1.3333,360.0000,264.0000,45.0000,113.0000,188.0000,190.0000,191.0000,132.6667,131.5000,96.0000,95.0000,81.0000,80.5000,1,1,1,0,1,208.0000,0,135.0000,135.0000,1
1,130379,66.0000,Non-Hispanic White,College graduate or above,5.0000,Married/Living with Partner,1,1.0000,No,33.5000,114.7000,101.8000,174.2000,1.0000,3.0000,3.0000,480.0000,214.0000,60.0000,125.3333,137.0000,135.0000,139.0000,117.0000,115.0000,78.6667,76.0000,72.0000,72.0000,1,1,1,0,1,300.0000,0,180.0000,135.0000,1
2,130380,44.0000,Other Hispanic,HS/GED or equivalent,1.4100,Married/Living with Partner,0,0.0000,Yes,29.7000,93.5000,69.4000,152.9000,2.0000,3.0000,1.0000,240.0000,187.0000,49.0000,156.0000,63.0000,90.0000,78.0000,109.0000,108.0000,78.3333,78.0000,81.3333,80.0000,1,0,0,0,1,1.0000,1,20.0000,21.9045,0
3,130386,34.0000,Mexican American,Some college or AA degree,1.3300,Married/Living with Partner,0,0.0000,No,30.2000,106.1000,90.6000,173.3000,1.0000,3.0000,2.0000,180.0000,183.0000,46.0000,100.0000,109.0000,111.0000,112.0000,115.0000,117.5000,73.6667,74.5000,62.3333,64.0000,0,1,1,0,1,104.0000,0,30.0000,6.9045,0
4,130394,51.0000,Non-Hispanic White,College graduate or above,5.0000,Married/Living with Partner,0,0.0000,No,24.4000,92.1000,76.7000,177.3000,2.0000,3.0000,1.0000,420.0000,183.0000,48.0000,88.0000,124.0000,120.0000,124.0000,110.6667,116.5000,68.0000,67.5000,79.6667,80.5000,0,1,0,0,1,24.0000,0,0.0000,120.0000,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3557,142301,80.0000,Non-Hispanic White,College graduate or above,1.2000,Widowed/Divorced/Separated,1,1.0000,No,30.5000,79.7000,82.2000,164.3000,1.0000,3.0000,2.0000,360.0000,138.0000,48.0000,110.0000,61.0000,67.0000,65.0000,140.4444,112.1667,75.3333,72.8333,69.7778,79.0000,1,0,1,0,1,1.0000,1,76.4020,21.9045,0
3558,142303,69.0000,Non-Hispanic White,HS/GED or equivalent,0.9800,Widowed/Divorced/Separated,0,0.0000,Yes,27.9000,111.0000,70.8000,159.2000,1.0000,3.0000,0.0000,360.0000,110.0000,34.0000,160.0000,45.0000,53.0000,50.0000,127.0000,126.0000,69.3333,68.5000,75.3333,75.0000,1,0,1,0,0,0.0000,0,840.0000,95.0000,1
3559,142305,76.0000,Mexican American,<HS,2.2500,Widowed/Divorced/Separated,1,1.0000,Yes,26.4000,89.0000,60.4000,151.4000,2.0000,3.0000,1.3333,480.0000,180.0000,51.0000,132.0000,92.0000,101.0000,97.0000,143.6667,146.0000,79.3333,78.5000,70.6667,70.5000,1,0,0,0,1,17.3333,1,80.0000,21.9045,0
3560,142308,50.0000,Other Hispanic,Some college or AA degree,1.9500,Married/Living with Partner,0,0.0000,No,26.4000,98.4000,79.3000,173.3000,2.0000,3.0000,2.0000,600.0000,166.0000,42.6667,112.6667,61.3333,122.6667,88.6667,108.0000,109.0000,69.3333,71.5000,62.6667,65.0000,0,1,0,0,1,8.0000,0,45.0000,24.6030,0


Configuration Settings

In [62]:
output_dir  = "../visualizations/models"
seed = 42
palette = "Set2"

# define feature columns
feature_cols = [
    'age', 'race_ethnicity', 'male', 'education', 'marital_status',
    'poverty_income_ratio', 'bmi', 'total_cholesterol_mgdl', 
    'physically_active', 'time_sitting', 'current_smoker', 'ever_smoker',
    'drinks_alcohol', 'avg_drinks_per_day', 'drink_frequency_past_year'
]

print(feature_cols)
print(len(feature_cols), 'input features')

# define output feature (target)
target_col = 'hypertension_risk'
print('distribution of target value:\n', ht_risk_df[target_col].value_counts())

['age', 'race_ethnicity', 'male', 'education', 'marital_status', 'poverty_income_ratio', 'bmi', 'total_cholesterol_mgdl', 'physically_active', 'time_sitting', 'current_smoker', 'ever_smoker', 'drinks_alcohol', 'avg_drinks_per_day', 'drink_frequency_past_year']
15 input features
distribution of target value:
 hypertension_risk
1    2145
0    1417
Name: count, dtype: int64


---

Additional Transformations Needed for SVM

In [63]:
# one-hot encode the categorical columns
ht_risk_model = ht_risk_df.copy()

X = ht_risk_model[feature_cols]
y = ht_risk_model[target_col]

categorical = X.select_dtypes(include=["object", "string"]).columns.tolist()
numeric = X.select_dtypes(include=[np.number]).columns.tolist()

# split data into training and testing sets (80/20 split) - stratified to preserve class balance
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=seed, stratify=y
)

# preprocessing: one-hot encode categoricals, scale numerics
scaler = StandardScaler()
encoder = OneHotEncoder(drop="first", handle_unknown="ignore", sparse_output=False)

# fit on training data
X_train_num = scaler.fit_transform(X_train[numeric])
X_train_cat = encoder.fit_transform(X_train[categorical])
X_train = np.hstack([X_train_num, X_train_cat])

# fit on test data
X_test_num = scaler.fit_transform(X_test[numeric])
X_test_cat = encoder.fit_transform(X_test[categorical])
X_test = np.hstack([X_test_num, X_test_cat])

# examine shape:
print(X_train.shape)  # rows = participants, columns = input risk-factor features
print(X_test.shape)  
print(y_train.shape) # one hypertension_risk label per participant
print(y_test.shape)

# print snapshots:
print(pd.DataFrame(X_train).head()) 
print(pd.DataFrame(X_test).head())  
print(pd.DataFrame(y_train).head()) 
print(pd.DataFrame(y_test).head())

(2849, 22)
(713, 22)
(2849,)
(713,)
       0       1       2       3       4       5       6       7       8      9       10      11     12     13     14     15     16     17     18     19     20     21
0 -0.0237  1.1182 -1.0886 -1.3247  0.1454  1.1424 -0.8969  2.9592  1.2507 0.2857 -0.0116  0.4830 0.0000 1.0000 0.0000 0.0000 0.0000 1.0000 0.0000 0.0000 0.0000 0.0000
1  0.7008  1.1182  1.2670 -0.7601 -0.7745  1.1424 -0.0059 -0.3379 -0.7995 0.2857 -0.3837  1.2888 0.0000 0.0000 1.0000 0.0000 1.0000 0.0000 0.0000 0.0000 0.0000 0.0000
2  0.5336 -0.8943  1.2670 -0.1687 -0.2419 -0.8754  0.5880 -0.3379 -0.7995 0.2857 -0.5698 -0.7686 0.0000 0.0000 1.0000 0.0000 1.0000 0.0000 0.0000 0.0000 0.0000 0.0000
3  0.3107  1.1182  1.0686 -0.8408  1.5737 -0.8754 -1.1939 -0.3379 -0.7995 0.2857 -0.0116  2.2352 0.0000 0.0000 1.0000 0.0000 1.0000 0.0000 0.0000 0.0000 0.0000 1.0000
4 -0.6368  1.1182 -0.6453 -0.5182  1.0411 -0.8754  0.5880 -0.3379  1.2507 0.2857 -0.0116 -0.7686 0.0000 0.0000 0.0000 0.0000 0.00

SVM Analysis

In [64]:
# use array of various costs to assess best fit 
cost_grid = [0.01, 0.1, 1, 10, 100]

# help prevent overfitting/ generalizes well by ensuring the model is evaluated on multiple distinct subsets of data
cross_validation = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)

results = {}        # filled in by each kernel cell
cost_curves = {}    # filled in by each kernel cell (for plotting)
 
# function to print cost sweep, best params, confusion matrix, accuracy; store results for each kernel
def evaluate(name, search):
    results = pd.DataFrame(search.cv_results_)
    curve = results.groupby("param_C")["mean_test_score"].max()
    cost_curves[name] = curve
 
    best = search.best_estimator_
    y_pred = best.predict(X_test)
    conf_matrix = confusion_matrix(y_test, y_pred)
    accuracy = accuracy_score(y_test, y_pred)
    results[name] = {"cv_accuracy": search.best_score_, "test_accuracy": accuracy,
                     "confusion_matrix": confusion_matrix, "best_params": search.best_params_}
 
    print(f"KERNEL: {name.upper()}")
    print("Accuracy by cost (best CV score at each cost):")
    for c, score in curve.items():
        print(f"    C = {float(c):>6}:  Cross Validation accuracy = {score:.4f}")
    print(f"\nBest params : {search.best_params_}")
    print(f"Cross Validation accuracy : {search.best_score_:.4f}")
    print(f"Test accuracy: {accuracy:.4f}")
    print("\nConfusion matrix (rows = actual, cols = predicted):")
    print("                 Pred:No   Pred:Risk")
    print(f"    Actual:No     {conf_matrix[0,0]:>7}   {conf_matrix[0,1]:>8}")
    print(f"    Actual:Risk   {conf_matrix[1,0]:>7}   {conf_matrix[1,1]:>8}")
    print("\n" + classification_report(y_test, y_pred,
                                       target_names=["No risk", "Risk"]))
 

1. Linear Kernel Anlysis

In [65]:
svc_linear = SVC(kernel="linear", random_state=seed)
grid_linear = GridSearchCV(svc_linear, {"C": cost_grid},
                           cv=cross_validation, scoring="accuracy", n_jobs=-1)
grid_linear.fit(X_train, y_train)
evaluate("linear", grid_linear)

KERNEL: LINEAR
Accuracy by cost (best CV score at each cost):
    C =   0.01:  Cross Validation accuracy = 0.7571
    C =    0.1:  Cross Validation accuracy = 0.7578
    C =    1.0:  Cross Validation accuracy = 0.7578
    C =   10.0:  Cross Validation accuracy = 0.7578
    C =  100.0:  Cross Validation accuracy = 0.7578

Best params : {'C': 0.1}
Cross Validation accuracy : 0.7578
Test accuracy: 0.7812

Confusion matrix (rows = actual, cols = predicted):
                 Pred:No   Pred:Risk
    Actual:No         182        102
    Actual:Risk        54        375

              precision    recall  f1-score   support

     No risk       0.77      0.64      0.70       284
        Risk       0.79      0.87      0.83       429

    accuracy                           0.78       713
   macro avg       0.78      0.76      0.76       713
weighted avg       0.78      0.78      0.78       713



2. RBF Kernel Analysis

In [66]:
svc_rbf = SVC(kernel="rbf", random_state=seed)
grid_rbf = GridSearchCV(svc_rbf, {"C": cost_grid, "gamma": ["scale", 0.01, 0.1]},
                        cv=cross_validation, scoring="accuracy", n_jobs=-1)
grid_rbf.fit(X_train, y_train)
evaluate("rbf", grid_rbf)

KERNEL: RBF
Accuracy by cost (best CV score at each cost):
    C =   0.01:  Cross Validation accuracy = 0.6023
    C =    0.1:  Cross Validation accuracy = 0.7532
    C =    1.0:  Cross Validation accuracy = 0.7613
    C =   10.0:  Cross Validation accuracy = 0.7578
    C =  100.0:  Cross Validation accuracy = 0.7487

Best params : {'C': 1, 'gamma': 0.01}
Cross Validation accuracy : 0.7613
Test accuracy: 0.7784

Confusion matrix (rows = actual, cols = predicted):
                 Pred:No   Pred:Risk
    Actual:No         176        108
    Actual:Risk        50        379

              precision    recall  f1-score   support

     No risk       0.78      0.62      0.69       284
        Risk       0.78      0.88      0.83       429

    accuracy                           0.78       713
   macro avg       0.78      0.75      0.76       713
weighted avg       0.78      0.78      0.77       713



3. Polynomial Kernel Analysis

In [67]:
svc_poly = SVC(kernel="poly", gamma="scale", random_state=seed)
grid_poly = GridSearchCV(svc_poly, {"C": cost_grid, "degree": [2, 3]},
                         cv=cross_validation, scoring="accuracy", n_jobs=-1)
grid_poly.fit(X_train, y_train)
evaluate("polynomial", grid_poly)

KERNEL: POLYNOMIAL
Accuracy by cost (best CV score at each cost):
    C =   0.01:  Cross Validation accuracy = 0.6072
    C =    0.1:  Cross Validation accuracy = 0.7073
    C =    1.0:  Cross Validation accuracy = 0.7452
    C =   10.0:  Cross Validation accuracy = 0.7385
    C =  100.0:  Cross Validation accuracy = 0.7371

Best params : {'C': 1, 'degree': 2}
Cross Validation accuracy : 0.7452
Test accuracy: 0.7588

Confusion matrix (rows = actual, cols = predicted):
                 Pred:No   Pred:Risk
    Actual:No         153        131
    Actual:Risk        41        388

              precision    recall  f1-score   support

     No risk       0.79      0.54      0.64       284
        Risk       0.75      0.90      0.82       429

    accuracy                           0.76       713
   macro avg       0.77      0.72      0.73       713
weighted avg       0.76      0.76      0.75       713



In [68]:
print(list(results.keys()))


[]


Kernel Comparison

In [69]:

summary = pd.DataFrame({
    k: {"Cross Validation accuracy": v["cv_accuracy"], "Test acc": v["test_accuracy"]}
    for k, v in results.items()
}).T.sort_values("Test acc", ascending=False)
print("Kernel Comparison (test-set accuracy)")
print(summary.round(4))
best_kernel = summary.index[0]
print(f"\nBest kernel: {best_kernel.upper()} "
      f"(test accuracy = {summary.loc[best_kernel, 'Test acc']:.4f})")

KeyError: 'Test acc'